### Célula 1 (Python): 

precisamos avisar ao Spark onde estão os arquivos. Usando o Python apenas para essa ponte inicial:

`no primeiro bloco:`
está criando o contexto do 1 notebook criado para dar sequencia nessa 2 fase (silver)

`no segundo bloco: `
usa a lib do pandas para ler o csv e criar um dataframe

`no terceiro bloco: `
utiliza o spark para puxar o dataframe criado com o pandas e transforma-lo numa view temporaria para usar depois como linguagem SQL.

In [0]:
import pandas as pd
import os

# Identifica a pasta do projeto (notebook 1 bronze) automaticamente
context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = context.notebookPath().get()
project_folder = "/Workspace/brazilian_ecommerce_olist"

# Lê o CSV dos clientes (que extraímos no Notebook 1)
df_pandas = pd.read_csv(f"{project_folder}/data/olist_customers_dataset.csv")

# Cria a View para podermos usar SQL daqui pra frente
spark.createDataFrame(df_pandas).createOrReplaceTempView("vw_customers_bronze")

### Célula 2 (SQL):
Agora, vamos usar o SQL para criar a tabela definitiva. Repare que no SQL a gente já aproveita para "limpar a casa":



In [0]:
%sql
-- Criando a tabela Delta na Camada Prata
CREATE OR REPLACE TABLE silver_olist_customers
USING DELTA
AS
SELECT 
    customer_id,
    customer_unique_id,
    CAST(customer_zip_code_prefix AS INT) as zip_code, -- Convertendo para número
    initcap(customer_city) as city,                   -- Primeira letra em maiúscula
    upper(customer_state) as state                    -- Estado sempre em maiúsculo
FROM vw_customers_bronze

### Célula 3 (Python):
Preparação dos Pedidos (Orders) 

In [0]:
import pandas as pd

# Lendo o CSV bruto de pedidos
df_orders_pd = pd.read_csv(f"{project_folder}/data/olist_orders_dataset.csv")

# Criando a view temporária para o SQL
spark.createDataFrame(df_orders_pd).createOrReplaceTempView("vw_orders_bronze")

### Refino da Tabela de Pedidos (Silver)

In [0]:
%sql
CREATE OR REPLACE TABLE silver_olist_orders
USING DELTA
AS
SELECT 
    order_id,
    customer_id,
    order_status,
    CAST(order_purchase_timestamp AS TIMESTAMP) as purchase_at,
    CAST(order_approved_at AS TIMESTAMP) as approved_at,
    CAST(order_delivered_carrier_date AS TIMESTAMP) as carrier_delivered_at,
    CAST(order_delivered_customer_date AS TIMESTAMP) as customer_delivered_at,
    CAST(order_estimated_delivery_date AS TIMESTAMP) as estimated_delivery_at
FROM vw_orders_bronze

### Célula 4 (Python):
Preparaçãos da Tabela de Itens (Order Itens) 


In [0]:
import pandas as pd

# Lendo o CSV bruto de pedidos
df_orders_pd = pd.read_csv(f"{project_folder}/data/olist_order_items_dataset.csv")

# Criando a view temporária para o SQL
spark.createDataFrame(df_orders_pd).createOrReplaceTempView("vw_order_items_bronze")

### Refino da Tabela de Itens

In [0]:
%sql
CREATE OR REPLACE TABLE silver_olist_order_items
USING DELTA
AS
SELECT 
    order_id,
    order_item_id,
    product_id,
    seller_id,
    CAST(shipping_limit_date AS TIMESTAMP) as shipping_limit_at,
    CAST(price AS DECIMAL(10,2)) as price,
    CAST(freight_value AS DECIMAL(10,2)) as freight_value
FROM (
    
    SELECT * FROM vw_order_items_bronze
)